In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.stats import genpareto
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from scipy.spatial.distance import pdist, squareform
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

DATA = Path("..") / "data"
FIGURAS = Path("..") / "figuras"
FIGURAS.mkdir(exist_ok=True)

# Umbrales del modelo (src/train.py) y periodo de entrenamiento de la red
U = {"O3": 63.0, "SO2": 20.2, "PM2.5": 38.0}
X = {"O3": 27.0, "SO2": 54.8, "PM2.5": 38.0}
INICIO, FIN = "2023-01-01", "2025-01-01"

In [2]:
ni = pd.read_csv(DATA / "datos_no_imputados.csv", low_memory=False)
for c in [c for c in ni.columns if " (" in c]:
    ni[c.split(" (")[0]] = ni[c.split(" (")[0]].fillna(ni[c])
ni["Fecha y hora"] = pd.to_datetime(ni["Fecha y hora"])

# Limpieza básica (Etapa 2)
for v in ["O3", "SO2", "PM2.5", "PM10"]:
    ni.loc[ni[v] < 0, v] = np.nan
ni.loc[ni["PM10"] >= 999, "PM10"] = np.nan
mal = ni["PM2.5"] > ni["PM10"]
ni.loc[mal, ["PM2.5", "PM10"]] = np.nan

obs = ni[(ni["Fecha y hora"] >= INICIO) & (ni["Fecha y hora"] < FIN)]
print(obs.shape)

(263111, 33)


In [ ]:
obs["Fecha y hora"] = pd.to_datetime(obs["Fecha y hora"])
obs["Dia_semana"] = obs["Fecha y hora"].dt.day_of_week
obs["Mes"] = obs["Fecha y hora"].dt.month
obs["Hora"] = obs["Fecha y hora"].dt.hour

In [21]:
obs.head()

,Fecha y hora,CO,NO,NO2,NOX,O3,PM10,PM2.5,PRS,RAINF,...,SO2 (ppb),SR (kW/m2),TOUT (ºC),WSR (km/h),WDR (azimutal),Fecha,Dia,Dia_semana,Hora,Mes
360114,2023-01-01 00:00:00,1.67,4.1,30.3,34.5,9.0,118.0,94.12,718.4,0.0,...,NaN,NaN,NaN,NaN,NaN,2023-01-01,1,6,0,1
360115,2023-01-01 01:00:00,1.70,5.0,28.9,34.1,9.0,97.0,96.79,710.7,0.0,...,NaN,NaN,NaN,NaN,NaN,2023-01-01,1,6,1,1
360116,2023-01-01 02:00:00,1.51,4.3,23.2,27.7,11.0,103.0,81.23,710.4,0.0,...,NaN,NaN,NaN,NaN,NaN,2023-01-01,1,6,2,1
360117,2023-01-01 03:00:00,1.30,3.8,20.9,24.9,11.0,83.0,49.83,710.1,0.0,...,NaN,NaN,NaN,NaN,NaN,2023-01-01,1,6,3,1
360118,2023-01-01 04:00:00,1.28,4.6,20.7,25.5,9.0,65.0,40.19,710.2,0.0,...,NaN,NaN,NaN,NaN,NaN,2023-01-01,1,6,4,1


In [70]:
var = "O3"
d = obs[(obs["TOUT"] > 40) & (obs["Hora"] < 16) & (obs["Hora"] > 12) & (obs["WSR"] < 5)] 
# d = obs
extra = 0
e = d[var][d[var] > U[var]+extra] - (U[var]+extra)
fila = {"excesos": len(e)}
if len(e) >= 30:                       # con menos de 30 excesos el ajuste no es confiable
    xi, _, sigma = genpareto.fit(e, floc=0)
    prob = 1 - genpareto.cdf(x=X[var]-extra, c=xi, scale=sigma)
    fila.update(xi=float(xi), sigma=float(sigma), prob=float(prob))
    
print(fila)

{'excesos': 17}


In [ ]:
features = ["PRS", "RAINF", "RH", "SR", "TOUT", "WSR", "Hora"]# "Mes", "Dia_Sema"
df = obs.copy()
for f in features:
    df[f] = pd.qcut(df[f], q=4, labels=False, duplicates='drop')

In [100]:
from itertools import product

var = "O3"
extra = [0, 10, 20]

# 1. Create Cartesian product grid DataFrame
unique_vals = [df[f].unique() for f in features]
df_new = pd.DataFrame(product(*unique_vals), columns=features)

# Initialize output columns
df_new["prob_0"] = 0.0
df_new["prob_10"] = 0.0
df_new["prob_20"] = 0.0

# 2. Iterate through each combination
for vals in df_new[features].itertuples(index=False):
    # Convert named tuple/row to list matching `vals`
    val_list = list(vals)
    
    # Filter original dataframe for the matching subset
    mask_df = (df[features] == val_list).all(axis=1)
    d = df[mask_df]
    
    # Filter df_new to update the current combination
    mask_df_new = (df_new[features] == val_list).all(axis=1)
    
    for ext in extra:
        
        # Calculate excesses
        threshold = U[var] + ext
        e = d[var][d[var] > threshold] - threshold
        
        if len(e) >= 30:
            xi, _, sigma = genpareto.fit(e, floc=0)
            prob = 1 - genpareto.cdf(x=X[var] - ext, c=xi, scale=sigma)
            
            # Update columns correctly in df_new
            df_new.loc[mask_df_new, f"prob_{ext}"] = prob

df_new.head()

,PRS,RAINF,RH,SR,TOUT,WSR,Hora,prob_0,prob_10,prob_20
0,2.0,0.0,1.0,0.0,0.0,1.0,0,0.0,0.0,0.0
1,2.0,0.0,1.0,0.0,0.0,1.0,1,0.0,0.0,0.0
2,2.0,0.0,1.0,0.0,0.0,1.0,2,0.0,0.0,0.0
3,2.0,0.0,1.0,0.0,0.0,1.0,3,0.0,0.0,0.0
4,2.0,0.0,1.0,0.0,0.0,0.0,0,0.0,0.0,0.0


In [119]:
df_new[df_new["prob_0"] > 0.0].sort_values(by="prob_0", ascending=False)

,PRS,RAINF,RH,SR,TOUT,WSR,Hora,prob_0,prob_10,prob_20
6274,1.0,0.0,0.0,2.0,3.0,3.0,2,0.378675,0.000000,0.000000
11370,0.0,0.0,0.0,3.0,3.0,2.0,2,0.358006,0.494180,0.722571
11362,0.0,0.0,0.0,3.0,3.0,1.0,2,0.337624,0.462718,0.698407
11074,0.0,0.0,0.0,0.0,3.0,3.0,2,0.319315,0.510203,0.798644
10362,0.0,0.0,1.0,3.0,3.0,1.0,2,0.315021,0.479069,0.000000
11274,0.0,0.0,0.0,2.0,3.0,3.0,2,0.313312,0.000000,0.000000
1366,2.0,0.0,0.0,3.0,3.0,0.0,2,0.308010,0.513310,0.691503
11374,0.0,0.0,0.0,3.0,3.0,3.0,2,0.295740,0.430110,0.700795
6271,1.0,0.0,0.0,2.0,3.0,2.0,3,0.290789,0.000000,0.000000
11270,0.0,0.0,0.0,2.0,3.0,2.0,2,0.289904,0.000000,0.000000


In [120]:
(df[  (df["PRS"]  ==  1)
   & (df["RAINF"] ==  0)
   & (df["RH"]    ==  0)
   & (df["SR"]    ==  2)
   & (df["TOUT"]  ==  3)
   & (df["WSR"]   ==  3)
   & (df["Hora"]  ==  2)
   & (df["O3"]    >  90.0)]["O3"].count() /
df[  (df["PRS"]   ==  1)
   & (df["RAINF"] ==  0)
   & (df["RH"]    ==  0)
   & (df["SR"]    ==  2)
   & (df["TOUT"]  ==  3)
   & (df["WSR"]   ==  3)
   & (df["Hora"]  ==  2)
   & (df["O3"]    >  63.0)]["O3"].count())

np.float64(0.25)

In [121]:
df[  (df["PRS"]   ==  1)
   & (df["RAINF"] ==  0)
   & (df["RH"]    ==  0)
   & (df["SR"]    ==  2)
   & (df["TOUT"]  ==  3)
   & (df["WSR"]   ==  3)
   & (df["Hora"]  ==  2)
   & (df["O3"]    >  63.0)]

,Fecha y hora,CO,NO,NO2,NOX,O3,PM10,PM2.5,PRS,RAINF,...,SO2 (ppb),SR (kW/m2),TOUT (ºC),WSR (km/h),WDR (azimutal),Fecha,Dia,Dia_semana,Hora,Mes
367138,2023-10-20 17:00:00,1.00,3.3,21.5,24.8,113.0,75.0,24.00,1.0,0.0,...,NaN,NaN,NaN,NaN,NaN,2023-10-20,20,4,2,10
370280,2023-02-28 17:00:00,1.12,4.4,16.0,20.4,82.0,142.0,24.00,1.0,0.0,...,NaN,NaN,NaN,NaN,NaN,2023-02-28,28,1,2,2
370304,2023-03-01 17:00:00,1.10,5.7,15.5,21.2,66.0,111.0,20.00,1.0,0.0,...,NaN,NaN,NaN,NaN,NaN,2023-03-01,1,2,2,3
370544,2023-03-11 17:00:00,1.28,5.6,15.6,21.1,66.0,118.0,34.00,1.0,0.0,...,NaN,NaN,NaN,NaN,NaN,2023-03-11,11,5,2,3
370904,2023-03-26 17:00:00,1.15,4.6,15.4,20.0,72.0,91.0,27.00,1.0,0.0,...,NaN,NaN,NaN,NaN,NaN,2023-03-26,26,6,2,3
377647,2023-01-01 17:00:00,1.29,4.8,24.3,29.1,69.0,54.0,12.00,1.0,0.0,...,NaN,NaN,NaN,NaN,NaN,2023-01-01,1,6,2,1
385109,2023-11-08 17:00:00,2.15,8.4,46.6,55.0,78.0,137.0,59.00,1.0,0.0,...,NaN,NaN,NaN,NaN,NaN,2023-11-08,8,2,2,11
475641,2023-03-11 17:00:00,1.61,3.7,11.9,15.6,71.0,92.0,31.00,1.0,0.0,...,NaN,NaN,NaN,NaN,NaN,2023-03-11,11,5,2,3
476001,2023-03-26 17:00:00,1.60,3.4,12.4,15.8,77.0,79.0,24.00,1.0,0.0,...,NaN,NaN,NaN,NaN,NaN,2023-03-26,26,6,2,3
476984,2023-05-06 16:00:00,0.84,2.8,20.9,23.8,88.0,106.0,55.00,1.0,0.0,...,NaN,NaN,NaN,NaN,NaN,2023-05-06,6,5,2,5
